In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys, pickle
ROOT = '/content/drive/MyDrive/CW_Folder_UG'
sys.path.insert(0, f'{ROOT}/Code')
from utils import *
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader
from scipy.stats import mode

MODELS_DIR = f'{ROOT}/Models'

with open(f'{ROOT}/Code/data_splits.pkl', 'rb') as f:
    d = pickle.load(f)
test_paths, test_labels = d['test_paths'], d['test_labels']
y_test = np.array(test_labels)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import shutil, os

LOCAL_ROOT = '/tmp/CW_data'

if not os.path.exists(LOCAL_ROOT):
    print("Copying dataset to local SSD.")
    shutil.copytree(ROOT, LOCAL_ROOT)
    print("Done.")
else:
    print("Local copy already exists, skipping.")

# Remap paths from Drive to local SSD
def remap(paths, src=ROOT, dst=LOCAL_ROOT):
    return [p.replace(src, dst) for p in paths]

test_paths = remap(test_paths)
print(f"Paths remapped to {LOCAL_ROOT}")

In [ ]:
with open(f'{ROOT}/Code/track1_results.pkl', 'rb') as f: t1 = pickle.load(f)
with open(f'{ROOT}/Code/track2_results.pkl', 'rb') as f: t2 = pickle.load(f)
with open(f'{ROOT}/Code/track3_results.pkl', 'rb') as f: t3 = pickle.load(f)
with open(f'{ROOT}/Code/track4_results.pkl', 'rb') as f: t4 = pickle.load(f)

t1_results = t1['results']
t2_results = t2['results']
t3_results = t3['results']
t4_results = t4['results']

best_t1 = t1['best_name']
best_t2 = t2['best_name']
best_t3 = t3['best_name']

print(f'Best Track 1: {best_t1}  ({t1_results[best_t1]["test_acc"]:.4f})')
print(f'Best Track 2: {best_t2}  ({t2_results[best_t2]["test_acc"]:.4f})')
print(f'Best Track 3: {best_t3}  ({t3_results[best_t3]["test_acc"]:.4f})')

In [ ]:
preds_t1 = t1_results[best_t1]['test_preds']
preds_t2 = t2_results[best_t2]['test_preds']
preds_t3 = t3_results[best_t3]['test_preds']

votes_5a    = np.stack([preds_t1, preds_t2], axis=1)
ensemble_5a = mode(votes_5a, axis=1).mode.flatten()
acc_5a = accuracy_score(y_test, ensemble_5a)
print(f'Ensemble 5a (T1+T2 Majority Vote): {acc_5a:.4f}')

In [ ]:
votes_5b    = np.stack([preds_t1, preds_t3], axis=1)
ensemble_5b = mode(votes_5b, axis=1).mode.flatten()
acc_5b = accuracy_score(y_test, ensemble_5b)
print(f'Ensemble 5b (T1+T3 Majority Vote): {acc_5b:.4f}')

In [ ]:
print('Loading fine-tuned models for softmax ensemble...')

def load_vgg16(path):
    m = models.vgg16(weights=None)
    m.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location=device))
    return m.eval().to(device)

def load_resnet50(path):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(2048, NUM_CLASSES)
    m.load_state_dict(torch.load(path, map_location=device))
    return m.eval().to(device)

vgg_ft = load_vgg16(f'{MODELS_DIR}/track4_VGG16_finetune.pt')
rn_ft  = load_resnet50(f'{MODELS_DIR}/track4_ResNet50_finetune.pt')

te_ds_224 = AgeDataset(test_paths, test_labels, get_transforms(CNN_SIZE))
te_dl_224 = DataLoader(te_ds_224, 32, shuffle=False, num_workers=2)

@torch.no_grad()
def get_softmax_probs(model, loader):
    probs = []
    for imgs, _ in loader:
        logits = model(imgs.to(device))
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.vstack(probs)

print('Getting VGG-16 softmax probabilities...')
probs_vgg = get_softmax_probs(vgg_ft, te_dl_224)
print('Getting ResNet-50 softmax probabilities...')
probs_rn  = get_softmax_probs(rn_ft,  te_dl_224)

avg_probs   = (probs_vgg + probs_rn) / 2
ensemble_5c = avg_probs.argmax(axis=1)
acc_5c = accuracy_score(y_test, ensemble_5c)
print(f'Ensemble 5c (VGG+ResNet Avg Softmax): {acc_5c:.4f}')

In [ ]:
results = {
    'T1+T2_MajVote':       {'test_acc': acc_5a, 'test_preds': ensemble_5a},
    'T1+T3_MajVote':       {'test_acc': acc_5b, 'test_preds': ensemble_5b},
    'VGG+RN50_AvgSoftmax': {'test_acc': acc_5c, 'test_preds': ensemble_5c},
}
print(f"{'Model':<25} {'Test Acc':>10}")
print('─' * 37)
for n, r in sorted(results.items(), key=lambda x: -x[1]['test_acc']):
    print(f"{n:<25} {r['test_acc']:>10.4f}")

best_name = max(results, key=lambda n: results[n]['test_acc'])
print(f"\nBest Track 5: {best_name}  (test acc={results[best_name]['test_acc']:.4f})")

In [ ]:
for name, r in results.items():
    evaluate(name, y_test, r['test_preds'])

In [ ]:
names  = list(results.keys())
accs   = [results[n]['test_acc'] for n in names]
colors = ['#4C72B0', '#55A868', '#C44E52']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, accs, color=colors)
ax.set_ylim(0, 1); ax.set_ylabel('Test Accuracy')
ax.set_title('Track 5 — Ensemble Model Comparison')
plt.xticks(rotation=20, ha='right')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.005, f'{acc:.3f}',
            ha='center', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
all_results = {}
for n, r in t1_results.items(): all_results[f'T1: {n}'] = r['test_acc']
for n, r in t2_results.items(): all_results[f'T2: {n}'] = r['test_acc']
for n, r in t3_results.items(): all_results[f'T3: {n}'] = r['test_acc']
for n, r in t4_results.items(): all_results[f'T4: {n}'] = r['test_acc']
all_results['T5: T1+T2 MajVote']    = acc_5a
all_results['T5: T1+T3 MajVote']    = acc_5b
all_results['T5: VGG+RN50 AvgProb'] = acc_5c

sorted_results = sorted(all_results.items(), key=lambda x: -x[1])
print(f"{'Rank':<5} {'Model':<42} {'Test Acc':>10}")
print('─' * 60)
for rank, (n, acc) in enumerate(sorted_results, 1):
    print(f'{rank:<5} {n:<42} {acc:>10.4f}')

In [ ]:
names  = [n for n, _ in sorted_results]
accs   = [a for _, a in sorted_results]
colors = ['#e74c3c' if 'T5' in n else
          '#3498db' if 'T4' in n else
          '#2ecc71' if 'T3' in n else
          '#f39c12' if 'T2' in n else '#95a5a6' for n in names]

fig, ax = plt.subplots(figsize=(12, max(8, len(names) * 0.35)))
ax.barh(names[::-1], accs[::-1], color=colors[::-1])
ax.set_xlabel('Test Accuracy')
ax.set_title('All Models — Full Cross-Track Comparison')
ax.axvline(max(accs), color='red', linestyle='--', alpha=0.5,
           label=f'Best: {max(accs):.4f}')
ax.legend()
for i, acc in enumerate(accs[::-1]):
    ax.text(acc + 0.002, i, f'{acc:.4f}', va='center', fontsize=7)

handles = [mpatches.Patch(color=c, label=l)
           for c, l in zip(['#95a5a6','#f39c12','#2ecc71','#3498db','#e74c3c'],
                           ['Track 1','Track 2','Track 3','Track 4','Track 5'])]
ax.legend(handles=handles, loc='lower right')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, preds) in zip(axes, [
    ('T1+T2 Majority Vote',  ensemble_5a),
    ('T1+T3 Majority Vote',  ensemble_5b),
    ('VGG+ResNet Avg Softmax', ensemble_5c)
]):
    plot_confusion_matrix(name, y_test, preds, ax=ax)
plt.suptitle('Track 5 — Ensemble Confusion Matrices', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
qualitative_grid(test_paths, test_labels, {
    'T1+T2 MajVote':   ensemble_5a,
    'T1+T3 MajVote':   ensemble_5b,
    'VGG+RN50 AvgProb': ensemble_5c,
}, n=8)

In [ ]:
with open(f'{ROOT}/Code/track5_results.pkl', 'wb') as f:
    pickle.dump({
        'results': {
            'T1+T2_MajVote':       {'test_acc': acc_5a, 'test_preds': ensemble_5a},
            'T1+T3_MajVote':       {'test_acc': acc_5b, 'test_preds': ensemble_5b},
            'VGG+RN50_AvgSoftmax': {'test_acc': acc_5c, 'test_preds': ensemble_5c},
        },
        'best_name': best_name,
        'all_results': all_results
    }, f)
print('Track 5 results saved')